## Inspect a trained transition model

Loads the checkpoint written by `src.training.train_transition` and rolls it out on complete held-out episodes
(seed 101), free-running from the first recorded state under the recorded actions, with the scene code $z = \mu$ of
each episode's map. Training windows are shorter than an episode, so the late steps test extrapolation:

1. Free-running trajectories against the recorded ones, on the scene map
2. Accumulated error: rollout against one-step (teacher-forced) error, and its spread across episodes
3. Collisions: predicted states inside walls, and how deep
4. Scene-code ablation: what the transition model takes from $z$

In [ ]:
import sys
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib import patheffects
from matplotlib.ticker import MaxNLocator
from scipy.ndimage import distance_transform_edt, map_coordinates

sys.path.insert(0, "..")  # repository root, notebook runs from notebooks/
from src.configs import ENV_CONFIGS
from src.datasets.rollouts import load_transitions, read_scene_maps, split_episodes
from src.envs.dmlab_autonomous_exploration import segment_from_numpy
from src.models.dds_vae import DDSVAE, normalize_maps, reparameterize
from src.models.transition import TransitionModel, apply_increment

# ── Config ───────────────────────────────────────────
@dataclass
class NotebookConfig:
    checkpoint: str = "../checkpoints/transition_dmlab/version_0/last.ckpt"
    scene_encoder: str = "../checkpoints/dds_vae_dmlab/version_0/stage2/last.ckpt"
    rollout_h5: str = "../data/DMLabRolloutDataset.h5"
    env: str = "dmlab"
    val_fraction: float = 0.05  # must match the training split
    seed: int = 101
    num_episodes: int = 64      # held-out episodes, rolled out over their full length
    device: str = "cuda"

# ── Model and data ───────────────────────────────────
cfg = NotebookConfig()
device = torch.device(cfg.device)
torch.set_grad_enabled(False)
torch.manual_seed(cfg.seed)
torch.set_float32_matmul_precision("highest"); torch.backends.cudnn.allow_tf32 = False  # as in training: limits accumulated rollout error
env = ENV_CONFIGS[cfg.env]()
model = TransitionModel.from_checkpoint(cfg.checkpoint, device)
train_length = torch.load(cfg.checkpoint, map_location="cpu")["hyper_parameters"]["optim"]["sequence_length"]

states, actions, _, valid = load_transitions(cfg.rollout_h5, env)
_, val_episodes = split_episodes(len(states), cfg.val_fraction, cfg.seed)
complete = val_episodes[valid[val_episodes].all(axis=1)]  # held-out episodes without simulator resets
episodes = complete[np.linspace(0, len(complete) - 1, cfg.num_episodes).astype(int)]
maps, bounds = read_scene_maps(cfg.rollout_h5, episodes, env)  # uint8 [E, H, W, 3], [E, 4]
s, a = (torch.from_numpy(x[episodes]).to(device) for x in (states, actions))  # [E, T, d_s], [E, T, d_a]
E, T = s.shape[:2]

encoder = DDSVAE.from_checkpoint(cfg.scene_encoder, device)
mu, logvar = encoder.encode(normalize_maps(torch.from_numpy(maps).to(device).permute(0, 3, 1, 2)))  # [E, d_z]

num_params = sum(p.numel() for p in model.parameters())
print(f"[INFO] Transition model: {num_params / 1e6:.2f}M parameters | d_z = {mu.shape[1]} | trained on windows of {train_length} states")
print(f"[INFO] {E} held-out episodes x {T} states ({len(complete)} complete held-out episodes available)")

# ── Style ────────────────────────────────────────────
PAPER, INK, SLATE, MIST, HAIRLINE, BLUE, RED = "#fbfbfb", "#191b21", "#626b7a", "#9ba6b7", "#dfe3ea", "#164be6", "#be4248"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "text.color": INK, "axes.labelcolor": SLATE, "xtick.color": SLATE, "ytick.color": SLATE,
    "axes.edgecolor": HAIRLINE, "axes.linewidth": 0.7, "axes.spines.top": False, "axes.spines.right": False,
    "font.family": "sans-serif", "font.sans-serif": ["Inter", "Arial", "DejaVu Sans"],
    "font.monospace": ["DM Mono", "DejaVu Sans Mono"], "font.size": 10.5, "axes.labelsize": 10,
    "axes.titlesize": 10.5, "axes.titlelocation": "left", "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "xtick.major.size": 0, "ytick.major.size": 0,
})

def style_axis(ax):
    """Quiet axis with mono tick numbers; linear axes get at most five ticks."""
    for axis, scale in ((ax.xaxis, ax.get_xscale()), (ax.yaxis, ax.get_yscale())):
        if scale == "linear": axis.set_major_locator(MaxNLocator(5))
    ax.tick_params(labelfontfamily="monospace")

# ── Rollouts, errors, and walls ──────────────────────
def rollout(z):
    """Free-running states [E, T, d_s] from each episode's first recorded state under its recorded actions, with scene codes z [E, d_z]."""
    return model.rollout(s[:, 0], z, a[:, :-1])[0]

def drift(predicted, target):
    """Position error (normalized units) and absolute heading error (degrees), each [E, T'], of predicted against target states [E, T', d_s]."""
    position = (predicted[..., :2] - target[..., :2]).norm(dim=-1)
    sin, cos = predicted[..., 2], predicted[..., 3]  # heading difference from the sine-cosine pairs, wrapped to [-pi, pi]
    heading = torch.atan2(sin * target[..., 3] - cos * target[..., 2], cos * target[..., 3] + sin * target[..., 2]).abs().rad2deg()
    return position.cpu().numpy(), heading.cpu().numpy()

# Signed distance to the nearest wall of each map in pixels: positive in free space, negative inside walls
free = [segment_from_numpy(m, wall_margin=0)[0] > 0 for m in maps]
signed = [distance_transform_edt(f) - distance_transform_edt(~f) for f in free]  # E x [H, W]
cell = (env.world_max - env.world_min) / maps.shape[1]  # world units per map pixel

def clearance(predicted):
    """Signed distance to the nearest wall (world units, negative inside walls) of normalized states [E, T, d_s] on each episode's map."""
    px, py = env.world_to_pixel(*env.denormalize_xy(predicted[..., 0], predicted[..., 1]), maps.shape[1])  # [E, T] pixel coordinates
    return cell * np.stack([map_coordinates(d, [py[i] - 0.5, px[i] - 0.5], order=1, mode="nearest") for i, d in enumerate(signed)])

# 1) Free-running rollout from s_0 with z = mu
states_hat = rollout(mu)  # [E, T, d_s]
position, heading = drift(states_hat, s)  # [E, T]

# 2) One-step predictions from the recorded history: teacher forcing, no accumulation
pred, _ = model(s[:, :-1], mu, a[:, :-1])  # standardized increments [E, T - 1, d_Delta]
next_hat = apply_increment(s[:, :-1], pred * model.delta_std + model.delta_mean)  # [E, T - 1, d_s]
one_position, one_heading = drift(next_hat, s[:, 1:])  # [E, T - 1]

# 3) Wall clearance of the predicted and recorded positions
clear_hat, clear_true = (clearance(x.cpu().numpy()) for x in (states_hat, s))  # [E, T]
print(f"[INFO] Median position error: {np.median(position[:, 50]):.3f} at step 50 | {np.median(position[:, train_length - 1]):.3f} at step {train_length - 1} | "
      f"{np.median(position[:, -1]):.3f} at step {T - 1}")

### Free-running trajectories

Recorded trajectory (white) and free-running prediction (blue) from the same first state, with the same actions.
The four episodes are the best, two intermediate, and worst final position error of the set; red dots are predicted
states inside walls of the map.

In [ ]:
final = position[:, -1]
shown = np.argsort(final)[np.linspace(0, E - 1, 4).astype(int)]  # best -> worst final position error
recorded, predicted = (env.denormalize_state(x.cpu().numpy())[..., :2] for x in (s, states_hat))  # [E, T, 2] world x, y
outline = [patheffects.withStroke(linewidth=2.6, foreground=INK)]

fig, axes = plt.subplots(1, 4, figsize=(13, 3.6), constrained_layout=True)
for ax, i in zip(axes, shown):
    x_min, y_min, x_max, y_max = bounds[i]
    ax.imshow(maps[i], extent=(x_min, x_max, y_min, y_max), interpolation="nearest")
    ax.plot(*recorded[i].T, color="white", lw=1.3, path_effects=outline)
    ax.plot(*predicted[i].T, color=BLUE, lw=1.3, path_effects=outline)
    ax.plot(*recorded[i, 0], "o", color="white", ms=4, mec=INK, mew=0.8)
    ax.scatter(*predicted[i][clear_hat[i] < 0].T, s=7, color=RED, linewidths=0, zorder=4)
    ax.set(xlim=(x_min, x_max), ylim=(y_min, y_max), title=f"episode {episodes[i]} · final error {final[i]:.2f}")
    ax.set_axis_off()
plt.show()

### Accumulated error

Position and heading error per rollout step against the one-step error of the same model fed the recorded history
(teacher forcing): the gap between the two is what accumulation adds. Lines are medians over episodes and bands the
interquartile range; the dashed line marks the training window. The last panel shows the per-episode position error
at fixed horizons, with the median in blue.

In [ ]:
def band(ax, steps, values, color, label):
    """Median and interquartile band of values [E, len(steps)] over episodes."""
    q1, median, q3 = np.quantile(values, [0.25, 0.5, 0.75], axis=0)
    ax.fill_between(steps, q1, q3, color=color, alpha=0.15, linewidth=0)
    ax.plot(steps, median, color=color, lw=1.6, label=label)

steps = np.arange(1, T)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), constrained_layout=True)
for ax, rolled, one_step, label in ((axes[0], position, one_position, "Position error (normalized units)"),
                                    (axes[1], heading, one_heading, "Heading error (degrees)")):
    band(ax, steps, rolled[:, 1:], BLUE, "free-running")
    band(ax, steps, one_step, SLATE, "one-step")
    ax.axvline(train_length - 1, color=MIST, lw=0.8, ls=(0, (3, 3)))
    ax.text(train_length - 1, 0.97, "training window ", transform=ax.get_xaxis_transform(), color=SLATE, ha="right", va="top", fontsize=8.5)
    ax.set(yscale="log", xlabel="Step", ylabel=label)
    ax.legend(loc="lower right", frameon=False, fontsize=8.5)
axes[0].set_title("Position drift"); axes[1].set_title("Heading drift")

horizons = [25, 50, 100, train_length - 1, T - 1]
jitter = np.random.default_rng(cfg.seed).uniform(-0.18, 0.18, E)
for k, h in enumerate(horizons):
    axes[2].scatter(k + jitter, position[:, h], s=9, color=MIST, linewidths=0)
    axes[2].hlines(np.median(position[:, h]), k - 0.3, k + 0.3, color=BLUE, lw=2)
axes[2].set(yscale="log", xticks=range(len(horizons)), xticklabels=horizons, xlabel="Rollout horizon (steps)", ylabel="Position error (normalized units)",
            title="Spread across episodes")
for ax in axes[:2]: style_axis(ax)
axes[2].tick_params(labelfontfamily="monospace")  # categorical horizons keep their fixed ticks
plt.show()

### Collisions

The transition model never sees the walls; it can only learn them through $z$ and the recorded history. Walls come
from segmenting each scene map and the signed distance to the nearest wall is read at every state (negative inside
walls, in world units). The recorded trajectories set the reference level of the segmentation itself. Left: share of episodes whose rollout has entered a wall by each step. Right: distribution of
the signed distance over all states.

In [ ]:
def first_hit(inside):
    """First step of each episode spent inside a wall, T if never; inside is [E, T] boolean."""
    return np.where(inside.any(1), inside.argmax(1), inside.shape[1])

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4), constrained_layout=True)
for clear, color, label in ((clear_true, MIST, "recorded"), (clear_hat, BLUE, "free-running")):
    axes[0].step(np.arange(T), (first_hit(clear < 0)[:, None] <= np.arange(T)).mean(0), where="post", color=color, lw=1.6, label=label)
low, high = np.quantile(np.concatenate([clear_hat.ravel(), clear_true.ravel()]), [0.001, 0.999])
for clear, color, label in ((clear_true, MIST, "recorded"), (clear_hat, BLUE, "free-running")):
    axes[1].hist(clear.ravel(), bins=100, range=(low, high), density=True, color=color, alpha=0.4, linewidth=0, label=label)
axes[1].axvline(0, color=INK, lw=0.8)
axes[0].axvline(train_length - 1, color=MIST, lw=0.8, ls=(0, (3, 3)))
axes[0].set(xlabel="Step", ylabel="Episodes that entered a wall", title=f"{(first_hit(clear_hat < 0) < T).mean():.0%} of rollouts hit a wall")
axes[1].set(xlabel="Signed distance to the nearest wall (world units)", ylabel="Density",
            title=f"{(clear_hat < 0).mean():.1%} of predicted states inside walls")
axes[0].yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
for ax in axes: ax.legend(loc="upper left" if ax is axes[0] else "upper right", frameon=False, fontsize=8.5); style_axis(ax)
axes[1].set_yticks([])
print(f"[INFO] Recorded states inside walls: {(clear_true < 0).mean():.2%} (segmentation reference) | predicted: {(clear_hat < 0).mean():.2%}")
plt.show()

### Scene-code ablation

The same held-out episodes rolled out with different scene codes. If the transition model uses $z$ to respect the
layout, the code of the episode's own map (blue) should give the lowest error and the fewest steps inside that
map's walls; the other codes are another episode's $\mu$, a prior sample $z \sim \mathcal{N}(0, I)$, and $z = 0$.
Training codes are posterior samples, so the sample of the own posterior is the in-distribution reference. Wall
steps are always counted on the episode's own map.

In [ ]:
variants = {"own μ": mu, "own posterior sample": reparameterize(mu, logvar), "another map's μ": mu.roll(1, 0),
            "prior sample": torch.randn_like(mu), "z = 0": torch.zeros_like(mu)}
errors, wall_share = [], []
for z in variants.values():
    hat = rollout(z)  # [E, T, d_s]
    errors.append(np.median(drift(hat, s)[0][:, -1]))
    wall_share.append(100 * (clearance(hat.cpu().numpy()) < 0).mean())

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.0), constrained_layout=True)
for ax, values, label in zip(axes, (errors, wall_share), ("Median position error at the last step", "Steps inside walls (%)")):
    ax.barh(list(variants), values, color=[BLUE] + [MIST] * (len(variants) - 1), height=0.6, linewidth=0)
    ax.invert_yaxis()
    for y, v in enumerate(values): ax.text(v, y, f" {v:.2f}", va="center", color=INK, fontsize=8.5, fontfamily="monospace")
    ax.set(xlabel=label); ax.xaxis.set_major_locator(MaxNLocator(5)); ax.tick_params(axis="x", labelfontfamily="monospace")
axes[1].axvline(100 * (clear_true < 0).mean(), color=SLATE, lw=0.8, ls=(0, (3, 3)))
axes[1].text(100 * (clear_true < 0).mean(), 1.0, " recorded", transform=axes[1].get_xaxis_transform(), color=SLATE, va="bottom", fontsize=8.5)
axes[1].tick_params(left=False, labelleft=False)
plt.show()